# 第6章　矩阵即变换 —— 从"表格"到"空间扭曲"

> **本章目标**：掌握矩阵乘法三种视角（代数/几何/神经网络层），手写全连接层+广播机制。
> **前置知识**：第 3 章（矩阵 shape）、第 5 章（点积）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("✅ 环境就绪")

---
## 6.1　矩阵与向量相乘——线性层核心

📐 **定义**：`**y** = **W** @ **x**`。`(m,n) @ (n,) → (m,)`。这就是 `nn.Linear` 的数学本质。

In [ ]:
W = np.array([[1, 2, 0, -1], [0, 1, 3, 2], [2, 0, 1, 0]])
x = np.array([2, 1, 3, 4])
y = W @ x
print(f"W @ x = {y}")
for i in range(3):
    assert abs(np.dot(W[i], x) - y[i]) < 1e-10
print("矩阵-向量乘法 = 逐行点积 ✅")

---
## 6.2　矩阵与矩阵乘法——批量变换

📐 **定义**：`**C** = **A** @ **B**`。`(m,k) @ (k,n) → (m,n)`。

In [ ]:
import time
A = np.random.randn(200, 300); B = np.random.randn(300, 400)

def slow_matmul(A, B):
    m,k=A.shape; _,n=B.shape; C=np.zeros((m,n))
    for i in range(m):
        for j in range(n):
            for t in range(k): C[i,j]+=A[i,t]*B[t,j]
    return C

t0=time.perf_counter(); C_slow=slow_matmul(A,B); t1=time.perf_counter()
C_fast=A@B; t2=time.perf_counter()
print(f"三重循环: {t1-t0:.2f}s  NumPy @: {t2-t1:.4f}s  加速: {(t1-t0)/(t2-t1):.0f}x")
assert np.allclose(C_slow, C_fast)

---
## 6.3　几何视角——矩阵扭曲空间

In [ ]:
def plot_grid_transform(W, ax, title=""):
    pts=np.stack(np.meshgrid(np.linspace(-2,2,10),np.linspace(-2,2,10)),axis=-1)
    orig=pts.reshape(-1,2); trans=orig@W.T
    ax.scatter(orig[:,0],orig[:,1],c='blue',alpha=0.3,s=10,label='original')
    ax.scatter(trans[:,0],trans[:,1],c='red',alpha=0.6,s=10,label='transformed')
    ax.axhline(0,color='gray',lw=0.5); ax.axvline(0,color='gray',lw=0.5)
    ax.set_xlim(-5,5); ax.set_ylim(-5,5); ax.set_aspect('equal'); ax.legend(fontsize=8)
    ax.set_title(title)

fig,axes=plt.subplots(1,3,figsize=(15,4))
plot_grid_transform(np.eye(2),axes[0],"Identity")
plot_grid_transform([[2,0],[0,1]],axes[1],"Stretch x")
plot_grid_transform([[1,0.5],[0,1]],axes[2],"Shear")
plt.tight_layout(); plt.show()

---
## 6.4　全连接层手撕

📐 `output = X @ W.T + b`

In [ ]:
def linear_layer(X, W, b):
    return X @ W.T + b

batch,d_in,d_out=32,784,256
X=np.random.randn(batch,d_in); W=np.random.randn(d_out,d_in)*.01; b=np.zeros(d_out)
out=linear_layer(X,W,b)
print(f"输入 {X.shape} → 输出 {out.shape}")
assert out.shape==(batch,d_out) and np.allclose(out[0],X[0]@W.T+b)
print("全连接层 ✅")

---
## 6.5　广播机制精讲

📐 **定义　广播（Broadcasting）**：从最后一维往前对齐，维度相等或一方为 1 即可。

In [ ]:
A=np.ones((3,4)); v=np.array([10,20,30,40])
print(f"(3,4)+(4,) = {(A+v).shape}")
col=np.array([[1],[2],[3]]); row=np.array([10,20,30,40])
print(f"(3,1)+(4,) = {(col+row).shape}")
print(col+row)
try: A+np.array([1,2,3])
except ValueError as e: print(f"广播失败: {e}")

---
## ✏️ 习题

> 在下方新建代码单元格作答。

1. （概念）`W @ x` 中 W 的行数和列数分别对应什么？
2. （概念）广播规则怎么描述？
3. （代码）手写 `batch_matmul`（三重循环），与 NumPy `@` 对比。
4. （代码）实现两层全连接网络：`relu(X@W1.T+b1) @ W2.T + b2`。

---

> 🔗 **章末钩子**：有些矩阵可逆——乘完还能"倒回去"。不可逆意味着什么？
> 预览：下一章——**逆矩阵与线性方程组**，以及著名的线性回归闭式解。

> 💡 **提示**：完成后运行 `Kernel → Restart & Run All` 验证。